## Milestone 1

In [ ]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("/content/train.csv")
opts = ["A", "B", "C", "D", "E"]

In [ ]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [ ]:
counts = train["answer"].value_counts()
print(counts)
ans1 = counts.max() + counts.min()
print("Q1:", ans1)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Q1: 814


In [ ]:
def clean(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()

vocab = set()
for p in train["prompt"]:
    vocab.update(clean(p))
print("Q2:", len(vocab))

Q2: 859


In [ ]:
# "Row ID 1" = the row whose id column == 1
row1 = train[train["id"] == 1].iloc[0]
words = clean(row1["prompt"])
filtered = [w for w in words if w not in ENGLISH_STOP_WORDS]
print("Q3:", len(filtered))

Q3: 13


In [ ]:
all_text = []
for c in ["prompt"] + opts:
    all_text += train[c].fillna("").astype(str).tolist()

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(all_text)
print("Q4:", len(tfidf.get_feature_names_out()))

Q4: 2762


In [ ]:
pv = tfidf.transform([str(row1["prompt"])])
av = tfidf.transform([str(row1["A"])])
print("Q5:", round(float(cosine_similarity(pv, av)[0][0]), 4))

Q5: 0.2328


In [ ]:
match = 0
for _, row in train.iterrows():
    pv = tfidf.transform([str(row["prompt"])])
    ov = tfidf.transform([str(row[o]) for o in opts])
    sims = cosine_similarity(pv, ov)[0]
    best = opts[int(np.argmax(sims))]
    if best == row["answer"]:
        match += 1
print("Q6:", round(100 * match / len(train), 2), "%")

Q6: 13.7 %


In [ ]:
def map3(truth, pred):
    pred = pred[:3]
    for i, p in enumerate(pred):
        if p == truth:
            return 1.0 / (i + 1)
    return 0.0

print("Q7:", map3("C", ["C", "A", "B"]))   # C is 1st -> 1.0
print("Q8:", map3("B", ["D", "B", "E"]))   # B is 2nd -> 0.5

Q7: 1.0
Q8: 0.5


In [ ]:
top3 = counts.index[:3].tolist()   # 1st, 2nd, 3rd most frequent
scores = [map3(a, top3) for a in train["answer"]]
print("Q9:", round(np.mean(scores), 4))

Q9: 0.4212


In [ ]:
scores = []
for _, row in train.iterrows():
    pv = tfidf.transform([str(row["prompt"])])
    ov = tfidf.transform([str(row[o]) for o in opts])
    sims = cosine_similarity(pv, ov)[0]
    ranked = [o for o, s in sorted(zip(opts, sims), key=lambda x: x[1], reverse=True)]
    scores.append(map3(row["answer"], ranked[:3]))
print("Q10:", round(np.mean(scores), 4))

Q10: 0.3119
